# 03 Sensitivity Analysis for Scoring Weights and Requirement Perturbations

This notebook requires eight condition-specific evidence-package JSON files. The default location is `Data/Evidence_Packages/`; an external directory can be selected in the configuration cell or through an environment variable.

In [ ]:
from pathlib import Path
import os
import json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT_OVERRIDE = os.getenv("ROBOT_ACTUATOR_REPO_ROOT") or None
EVIDENCE_DIR_OVERRIDE = os.getenv("ROBOT_ACTUATOR_EVIDENCE_DIR") or None
OUTPUT_DIR_OVERRIDE = os.getenv("ROBOT_ACTUATOR_OUTPUT_DIR") or None

def resolve_repo_root(override=None):
    if override:
        root = Path(override).expanduser().resolve()
        if not (root / "Code").is_dir() or not (root / "Data").is_dir():
            raise FileNotFoundError(f"Invalid repository root: {root}")
        return root
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "Code").is_dir() and (candidate / "Data").is_dir():
            return candidate
    raise FileNotFoundError(
        "Unable to locate the repository root automatically. Start Jupyter from the repository root or Code/, "
        "or set ROBOT_ACTUATOR_REPO_ROOT."
    )

REPO_ROOT = resolve_repo_root(REPO_ROOT_OVERRIDE)
EVIDENCE_DIR = (
    Path(EVIDENCE_DIR_OVERRIDE).expanduser().resolve()
    if EVIDENCE_DIR_OVERRIDE
    else REPO_ROOT / "Data" / "Evidence_Packages"
)
OUTPUT_DIR = (
    Path(OUTPUT_DIR_OVERRIDE).expanduser().resolve()
    if OUTPUT_DIR_OVERRIDE
    else REPO_ROOT / "Reproduced_Outputs" / "03_scoring_sensitivity"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_MONTE_CARLO=int(os.getenv("ROBOT_ACTUATOR_MONTE_CARLO", "10000"))
DEMAND_SIGMA=.08; SAFETY_SIGMA=.05; rng=np.random.default_rng(42)
SCENARIOS={"balanced":[.35,.35,.20,.10],"lightweight_priority":[.20,.50,.25,.05],"efficiency_priority":[.25,.25,.15,.35],"safety_redundancy_priority":[.55,.25,.10,.10]}
W_NAMES=["safety_margin","mass","volume","efficiency"]
paths=sorted(EVIDENCE_DIR.glob("03_xgboost_shap_physics_score_*.json"))
if not paths:
    raise FileNotFoundError(
        f"No files matching 03_xgboost_shap_physics_score_*.json were found in {EVIDENCE_DIR}."
        "Place the complete evidence-package set in that directory or set ROBOT_ACTUATOR_EVIDENCE_DIR."
    )
packages=[json.loads(p.read_text(encoding="utf-8")) for p in paths]
if len(packages) != 8:
    print(f"Warning: {len(packages)} task conditions were detected; the manuscript experiment specifies eight. Verify the evidence-package directory.")
print("Evidence-package directory:", EVIDENCE_DIR, "; task conditions:", len(packages))

In [ ]:
def number(x):
    try: return float(x)
    except: return np.nan
def margin(rec,demand,safety):
    comparisons=(rec.get("physics_check") or {}).get("comparisons") or {}
    values=[]
    for v in comparisons.values():
        if isinstance(v,dict):
            cap,req=number(v.get("capability")),number(v.get("requirement"))
            if np.isfinite(cap) and np.isfinite(req) and req>0: values.append(cap/(req*demand*safety))
    return min(values) if values else 0.
def spec(specs,words):
    for k,v in (specs or {}).items():
        if any(x in str(k).lower() for x in words):
            z=number(v)
            if np.isfinite(z): return z
    return np.nan
def benefit(a,invert=False):
    a=np.asarray(a,float); good=np.isfinite(a)
    if not good.any(): return np.full(len(a),.5)
    a=np.where(good,a,np.nanmedian(a[good])); lo,hi=a.min(),a.max()
    z=np.full(len(a),.5) if math.isclose(lo,hi) else (a-lo)/(hi-lo)
    return 1-z if invert else z
def rank(records,demand,safety,w):
    z=[]
    for r in records:
        m=margin(r,demand,safety)
        if m>=1:  # Hard constraint: infeasible candidates are excluded from scoring.
            s=r.get("specs") or {}
            z.append({"record":r,"margin":m,"mass":spec(s,["mass", "\u8d28\u91cf"]),"volume":spec(s,["volume", "\u4f53\u79ef"]),"eff":spec(s,["efficiency", "\u6548\u7387"])})
    if not z:return []
    d=pd.DataFrame([{k:v for k,v in x.items() if k!="record"} for x in z])
    scores=np.column_stack([np.minimum(d.margin/2,1),benefit(d.mass,True),benefit(d.volume,True),benefit(d.eff)]) @ np.asarray(w)
    for x,v in zip(z,scores):x["score"]=float(v)
    return sorted(z,key=lambda x:x["score"],reverse=True)
def simulate(pkg,scenario,w):
    records=pkg.get("candidate_records") or []; baseline=rank(records,1,1,w)
    baseline_id=baseline[0]["record"].get("candidate_id") if baseline else None; result=[]
    for i in range(N_MONTE_CARLO):
        demand=max(.1,rng.normal(1,DEMAND_SIGMA)); safety=max(.1,rng.normal(1,SAFETY_SIGMA)); weights=rng.dirichlet(np.maximum(np.asarray(w)*80,.5))
        r=rank(records,demand,safety,weights); pick=r[0]["record"] if r else {}
        brank=next((j+1 for j,x in enumerate(r) if x["record"].get("candidate_id")==baseline_id),np.nan)
        result.append({"condition":(pkg.get("joint_requirements") or {}).get("condition","unknown"),"scenario":scenario,"run":i,
          "demand_multiplier":demand,"safety_multiplier":safety,"feasible_count":len(r),"baseline_id":baseline_id,
          "selected_id":pick.get("candidate_id"),"selected_drive_type":pick.get("drive_type"),"baseline_rank":brank,
          **{f"w_{n}":float(x) for n,x in zip(W_NAMES,weights)}})
    return pd.DataFrame(result)


In [ ]:
runs=[]
for pkg in packages:
    for name,w in SCENARIOS.items():
        print((pkg.get("joint_requirements") or {}).get("condition"),name); runs.append(simulate(pkg,name,w))
mc=pd.concat(runs,ignore_index=True); mc["retained"]=mc.selected_id.eq(mc.baseline_id)
summary=mc.groupby(["condition","scenario"]).agg(top1_retention=("retained","mean"),rank_flip_probability=("retained",lambda x:1-x.mean()),
    average_baseline_rank=("baseline_rank","mean"),feasible_count=("feasible_count","mean")).reset_index()
ratios=mc.groupby(["condition","scenario","selected_drive_type"]).size().rename("count").reset_index(); ratios["selection_ratio"]=ratios.groupby(["condition","scenario"])["count"].transform(lambda x:x/x.sum())
flips=mc[~mc.retained].sort_values("run").groupby(["condition","scenario"],as_index=False).first()
mc.to_csv(OUTPUT_DIR/"monte_carlo_runs.csv",index=False,encoding="utf-8-sig"); summary.to_csv(OUTPUT_DIR/"stability_summary.csv",index=False,encoding="utf-8-sig")
ratios.to_csv(OUTPUT_DIR/"drive_type_selection_ratio.csv",index=False,encoding="utf-8-sig"); flips.to_csv(OUTPUT_DIR/"first_rank_flip_cases.csv",index=False,encoding="utf-8-sig")
display(summary); display(ratios.head(20))
p=summary.pivot(index="condition",columns="scenario",values="top1_retention"); fig,ax=plt.subplots(figsize=(9,max(3,.6*len(p))))
im=ax.imshow(p,vmin=0,vmax=1,cmap="YlGn"); ax.set_xticks(range(len(p.columns)),p.columns,rotation=25,ha="right"); ax.set_yticks(range(len(p.index)),p.index)
for i in range(len(p.index)):
 for j in range(len(p.columns)): ax.text(j,i,f"{p.iloc[i,j]:.2f}",ha="center",va="center")
fig.colorbar(im,ax=ax,label="Top-1 retention rate"); plt.tight_layout(); plt.savefig(OUTPUT_DIR/"retention_heatmap.png",dpi=220); plt.show()
print("Completed. Outputs report retention rates, rank-flip probabilities, mean ranks, feasible-candidate counts, and the requirement/weight conditions associated with the first rank flip.", OUTPUT_DIR)
